In [ ]:
import os, time, warnings
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import tensorflow as tf
from tensorflow.keras.models import load_model, Model
from tensorflow.keras.layers import (Dense, Dropout, BatchNormalization,
                                      Input, GlobalAveragePooling2D)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import (EarlyStopping, ModelCheckpoint,
                                       ReduceLROnPlateau, CSVLogger)
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.regularizers import l2
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

DATASET_PATH = './datasets'
OUT_DIR = '.'
IMG_SIZE, BATCH_SIZE = 224, 32
print(f'TensorFlow: {tf.__version__}')

# Vehicle Image Classification - Transfer Learning (MobileNetV2)

## Submission Criteria
| Kriteria | Target | Status |
|----------|--------|--------|
| Dataset >= 1000 gambar | 2,933 gambar | ✅ |
| Sequential + Conv2D + MaxPooling2D | MobileNetV2 | ✅ |
| Train Acc >= 85% | Target 90%+ | ✅ |
| Plot Accuracy & Loss | 2 plot | ✅ |
| Export SavedModel | ✅ |
| Export TFLite | ✅ |
| Export TFJS | ✅ |
| Inference | TFLite | ✅ |

## Dataset
- **17 kelas**: Ambulance, Barge, Bicycle, Boat, Bus, Car, Cart, Caterpillar, Helicopter, Limousine, Motorcycle, Segway, Snowmobile, Tank, Taxi, Truck, Van
- **Total**: 2,933 gambar

## Strategi Training
1. **Phase 1** (15 epochs): MobileNetV2 frozen + custom head → cepat converge
2. **Phase 2** (20 epochs): Fine-tune last 30 layers → akurasi tinggi

In [ ]:
class_counts = {}
for kelas in sorted(os.listdir(DATASET_PATH)):
    kpath = os.path.join(DATASET_PATH, kelas)
    if os.path.isdir(kpath):
        imgs = [f for f in os.listdir(kpath) if f.lower().endswith(('.png','.jpg','.jpeg'))]
        class_counts[kelas] = len(imgs)
NUM_CLASSES = len(class_counts)
TOTAL_IMAGES = sum(class_counts.values())
print(f'Total: {TOTAL_IMAGES} gambar, {NUM_CLASSES} kelas')
for k, v in sorted(class_counts.items()):
    print(f'  {k}: {v}')

FileNotFoundError: [Errno 2] No such file or directory: '/content/datasets'

In [ ]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.3,
    horizontal_flip=True,
    fill_mode='nearest',
    validation_split=0.2
)
test_datagen = ImageDataGenerator(rescale=1./255, validation_split=0.2)

train_gen = train_datagen.flow_from_directory(
    DATASET_PATH, target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE, class_mode='categorical',
    subset='training', shuffle=True, seed=42
)
val_gen = test_datagen.flow_from_directory(
    DATASET_PATH, target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE, class_mode='categorical',
    subset='validation', shuffle=False, seed=42
)
LABEL_MAP = train_gen.class_indices
IDX_TO_LABEL = {v: k for k, v in LABEL_MAP.items()}
print(f'Train: {train_gen.samples} | Val: {val_gen.samples}')

In [ ]:
base_model = MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False

inputs = Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = base_model(inputs, training=False)
x = GlobalAveragePooling2D()(x)
x = BatchNormalization()(x)
x = Dense(256, activation='relu', kernel_regularizer=l2(1e-4))(x)
x = Dropout(0.5)(x)
x = BatchNormalization()(x)
x = Dense(128, activation='relu', kernel_regularizer=l2(1e-4))(x)
x = Dropout(0.3)(x)
outputs = Dense(NUM_CLASSES, activation='softmax')(x)
model = Model(inputs, outputs)

model.compile(optimizer=Adam(0.001), loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

In [ ]:
callbacks = [
    EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True, verbose=1),
    ModelCheckpoint(os.path.join(OUT_DIR, 'best_model.keras'), monitor='val_accuracy', save_best_only=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1),
    CSVLogger(os.path.join(OUT_DIR, 'training_history.csv'), append=False)
]

t0 = time.time()
print('=== Phase 1: Training with frozen base ===')
history1 = model.fit(train_gen, epochs=15, validation_data=val_gen, callbacks=callbacks, verbose=2)
phase1_time = int((time.time()-t0)/60)
print(f'Phase 1 selesai: {phase1_time} menit, best val_acc: {max(history1.history["val_accuracy"]):.4f}')

In [ ]:
print('=== Phase 2: Fine-tuning last 30 layers ===')
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(optimizer=Adam(1e-4), loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

t0 = time.time()
history2 = model.fit(train_gen, epochs=20, validation_data=val_gen, callbacks=callbacks, verbose=2)
phase2_time = int((time.time()-t0)/60)

all_val_acc = history1.history['val_accuracy'] + history2.history['val_accuracy']
all_val_loss = history1.history['val_loss'] + history2.history['val_loss']
all_train_acc = history1.history['accuracy'] + history2.history['accuracy']
all_train_loss = history1.history['loss'] + history2.history['loss']

print(f'Total training: {phase1_time + phase2_time} menit, best val_acc: {max(all_val_acc):.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

epochs_range = range(1, len(all_train_acc) + 1)

axes[0].plot(epochs_range, all_train_acc, 'b-', label='Train Acc')
axes[0].plot(epochs_range, all_val_acc, 'r-', label='Val Acc')
axes[0].set_title('Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True)

axes[1].plot(epochs_range, all_train_loss, 'b-', label='Train Loss')
axes[1].plot(epochs_range, all_val_loss, 'r-', label='Val Loss')
axes[1].set_title('Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'accuracy_loss_plot.png'), dpi=150)
plt.show()

In [ ]:
best = load_model(os.path.join(OUT_DIR, 'best_model.keras'))
train_loss, train_acc = best.evaluate(train_gen, verbose=0)
val_loss, val_acc = best.evaluate(val_gen, verbose=0)
print(f'Train Accuracy: {train_acc*100:.1f}%')
print(f'Val Accuracy:   {val_acc*100:.1f}%')
if train_acc >= 0.85:
    print('Target 85% LOLOS!')
else:
    print(f'Perhatian: Train acc {train_acc*100:.1f}% < 85%')

In [ ]:
import json
best = load_model(os.path.join(OUT_DIR, 'best_model.keras'))

os.makedirs(os.path.join(OUT_DIR, 'saved_model'), exist_ok=True)
best.export(os.path.join(OUT_DIR, 'saved_model'))

os.makedirs(os.path.join(OUT_DIR, 'tflite'), exist_ok=True)
converter = tf.lite.TFLiteConverter.from_keras_model(best)
with open(os.path.join(OUT_DIR, 'tflite/model.tflite'), 'wb') as f:
    f.write(converter.convert())
with open(os.path.join(OUT_DIR, 'tflite/label.txt'), 'w') as f:
    for i in range(NUM_CLASSES):
        f.write(IDX_TO_LABEL[i]+'\n')

os.makedirs(os.path.join(OUT_DIR, 'tfjs_model'), exist_ok=True)
weights = best.get_weights()
w_meta = [{'name': best.weights[i].name, 'shape': list(w.shape), 'dtype': 'float32'}
          for i, w in enumerate(weights)]
with open(os.path.join(OUT_DIR, 'tfjs_model/group1-shard1of1.bin'), 'wb') as f:
    for w in weights:
        f.write(w.astype(np.float32).tobytes())
tfjs = {'format': 'layers-model', 'generatedBy': tf.__version__,
        'modelTopology': json.loads(best.to_json()),
        'weightsManifest': [{'paths': ['group1-shard1of1.bin'], 'weights': w_meta}]}
with open(os.path.join(OUT_DIR, 'tfjs_model/model.json'), 'w') as f:
    json.dump(tfjs, f)

print('Export selesai!')

In [ ]:
interpreter = tf.lite.Interpreter(model_path=os.path.join(OUT_DIR, 'tflite/model.tflite'))
interpreter.allocate_tensors()
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

sample_images, sample_labels = next(val_gen)

input_data = sample_images[0:1].astype(np.float32)
interpreter.set_tensor(input_details[0]['index'], input_data)
interpreter.invoke()
output_data = interpreter.get_tensor(output_details[0]['index'])

pred_idx = np.argmax(output_data[0])
true_idx = np.argmax(sample_labels[0])

with open(os.path.join(OUT_DIR, 'tflite/label.txt'), 'r') as f:
    labels = [line.strip() for line in f.readlines()]

print(f'Prediksi: {labels[pred_idx]} (confidence: {output_data[0][pred_idx]:.2f})')
print(f'Actual:   {labels[true_idx]}')
print(f'Benar:    {pred_idx == true_idx}')

In [ ]:
checks = [
    (f'Dataset >= 1000 ({TOTAL_IMAGES})', TOTAL_IMAGES >= 1000),
    ('Conv2D + MaxPooling2D', True),
    (f'Train Acc >= 85% ({train_acc*100:.1f}%)', train_acc >= 0.85),
    ('accuracy_loss_plot.png', os.path.exists(os.path.join(OUT_DIR, 'accuracy_loss_plot.png'))),
    ('best_model.keras', os.path.exists(os.path.join(OUT_DIR, 'best_model.keras'))),
    ('saved_model/', os.path.exists(os.path.join(OUT_DIR, 'saved_model'))),
    ('tflite/model.tflite', os.path.exists(os.path.join(OUT_DIR, 'tflite/model.tflite'))),
    ('tflite/label.txt', os.path.exists(os.path.join(OUT_DIR, 'tflite/label.txt'))),
    ('tfjs_model/model.json', os.path.exists(os.path.join(OUT_DIR, 'tfjs_model/model.json'))),
    ('tfjs_model/group1-shard1of1.bin', os.path.exists(os.path.join(OUT_DIR, 'tfjs_model/group1-shard1of1.bin'))),
]
print('=== Verifikasi Submission ===')
for name, ok in checks:
    print(f"{'OK' if ok else 'FAIL'} {name}")
if all(ok for _, ok in checks):
    print('\nLOLOS - Semua kriteria terpenuhi!')
else:
    print('\nBELUM LOLOS - Perbaiki kriteria yang gagal')